# Solution 02: Generate Classification Dataset with LLM

In [ ]:
import json, os
from openai import OpenAI
from collections import Counter
from tqdm import tqdm

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "raw_security_reports.json")) as f:
    reports = json.load(f)

api_key = "your-api-key-here"  # Replace with your actual OpenAI key
client = OpenAI(api_key=api_key)

ALL_ATTACK_LABELS = [
    "ransomware", "phishing", "apt_attack", "ddos",
    "data_breach", "supply_chain_attack", "zero_day"
]
print(f"✓ Loaded {len(reports)} reports")

## Part 1: Call LLM and Parse JSON Response

In [2]:
SYSTEM_PROMPT = """You are a cybersecurity analyst. Classify the given report.

Available attack type labels:
ransomware, phishing, apt_attack, ddos, data_breach, supply_chain_attack, zero_day

Return a JSON object with:
- "true_labels": list of labels that APPLY to this report (1-3 labels)
- "false_labels": list of 2-3 labels that clearly do NOT apply

Return ONLY the JSON object, nothing else."""


def call_llm_cls(client, text, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


test_text = reports[0]['text']
cls_result = call_llm_cls(client, test_text)

assert isinstance(cls_result, dict)
assert 'true_labels' in cls_result and 'false_labels' in cls_result
assert 'ransomware' in cls_result['true_labels']
print(f"✓ true_labels:  {cls_result['true_labels']}")
print(f"  false_labels: {cls_result['false_labels']}")

✓ true_labels:  ['ransomware', 'apt_attack', 'zero_day']
  false_labels: ['phishing', 'ddos', 'data_breach']


## Part 2: Implement `build_gliclass_example`

In [3]:
def build_gliclass_example(client, text):
    data = call_llm_cls(client, text)
    true_labels = [l for l in data.get('true_labels', []) if l in ALL_ATTACK_LABELS]
    false_labels = [l for l in data.get('false_labels', []) if l in ALL_ATTACK_LABELS]
    all_labels = list(set(true_labels + false_labels))
    if not true_labels or not all_labels:
        return None
    return {"text": text, "all_labels": all_labels, "true_labels": true_labels}


ex = build_gliclass_example(client, reports[1]['text'])  # phishing
assert ex is not None
assert set(ex['true_labels']).issubset(set(ex['all_labels']))
assert 'phishing' in ex['true_labels']
print(f"✓ build_gliclass_example works")
print(f"  all_labels:  {ex['all_labels']}")
print(f"  true_labels: {ex['true_labels']}")

✓ build_gliclass_example works
  all_labels:  ['data_breach', 'ransomware', 'phishing', 'ddos']
  true_labels: ['phishing']


## Part 3: Generate Full Dataset

In [4]:
def build_cls_dataset(client, reports):
    dataset = []
    for r in tqdm(reports, desc="Annotating classification"):
        try:
            ex = build_gliclass_example(client, r['text'])
            if ex:
                dataset.append(ex)
        except Exception as e:
            print(f"  Error on report {r['id']}: {e}")
    return dataset


cls_dataset = build_cls_dataset(client, reports)

assert len(cls_dataset) >= 15
all_true_labels = set()
for ex in cls_dataset:
    assert set(ex['true_labels']).issubset(set(ex['all_labels']))
    for label in ex['all_labels']:
        assert label in ALL_ATTACK_LABELS
    all_true_labels.update(ex['true_labels'])
assert len(all_true_labels) >= 4

save_path = os.path.normpath(os.path.join(fixtures, "..", "expected", "cls_dataset.json"))
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(cls_dataset, f, indent=2)

label_counts = Counter(l for ex in cls_dataset for l in ex['true_labels'])
print(f"✓ Generated {len(cls_dataset)} classification examples")
print(f"  Attack types: {sorted(all_true_labels)}")
print(f"  Distribution: {dict(label_counts.most_common())}")
print(f"  Saved to fixtures/expected/cls_dataset.json")

Annotating classification:   0%|          | 0/20 [00:00<?, ?it/s]

Annotating classification:   5%|▌         | 1/20 [00:01<00:30,  1.59s/it]

Annotating classification:  10%|█         | 2/20 [00:02<00:24,  1.33s/it]

Annotating classification:  15%|█▌        | 3/20 [00:03<00:21,  1.24s/it]

Annotating classification:  20%|██        | 4/20 [00:05<00:23,  1.47s/it]

...

Annotating classification:  90%|█████████ | 18/20 [00:22<00:02,  1.29s/it]

Annotating classification:  95%|█████████▌| 19/20 [00:24<00:01,  1.28s/it]

Annotating classification: 100%|██████████| 20/20 [00:25<00:00,  1.31s/it]

Annotating classification: 100%|██████████| 20/20 [00:25<00:00,  1.28s/it]

✓ Generated 19 classification examples
  Attack types: ['apt_attack', 'data_breach', 'ddos', 'phishing', 'ransomware', 'supply_chain_attack', 'zero_day']
  Distribution: {'apt_attack': 9, 'zero_day': 9, 'data_breach': 7, 'ransomware': 6, 'phishing': 5, 'supply_chain_attack': 5, 'ddos': 3}
  Saved to fixtures/expected/cls_dataset.json
